# Suite EVL — Evaluators

`Evaluators.OfAny(expression, scope)` computes an expression's value with the variables in `scope` bound, and
`Evaluators.OfLiteral`, `OfOperation`, `OfVariable` and `OfLet` evaluate one kind. Evaluation is three-valued (absent
and incomparable values are unknown, and logic follows Kleene), never coerces native types, and evaluates only the core
operations.

In [ ]:
import math

from mbse_schemas.Framework import Evaluators as V, Expressions as E, Proxies, Schemas as S
from support import raises, text

op = E.OfOperation.Builder().name("f").arguments(1, "x").create()


def ev(expression, **scope):
    """Evaluates `expression` with `scope` bound."""
    return V.OfAny(expression, scope)

## EVL-01 · Evaluating literals, variables, lets, `get` and `has`

In [ ]:
Person = S.OfObject.Builder().properties(text("name"), text("age", int)).create()
Proxies.register("Person", Person)
ann = Proxies.Builders.Person().name("Ann").age(30).create()
nobody = Proxies.Builders.Person().create()
this = E.variable("this")
assert ev(3) == 3 and ev(E.literal("x")) == "x" and ev(this, this=ann) is ann
assert ev(this.age, this=ann) == 30 and ev(this.age, this=nobody) is None  # absent: unknown
assert ev(this.has("age"), this=ann) is True and ev(this.has("age"), this=nobody) is False
assert ev(this.age, this=None) is None and ev(this.has("age"), this=None) is None  # unknown object
assert ev(E.let_("a", this.age, E.variable("a").add(1)), this=ann) == 31
assert ev(E.let_("a", 1, E.let_("a", 2, E.variable("a")))) == 2  # the innermost binding wins
assert ev(E.let_("a", 1, E.variable("a").add(E.let_("a", 2, E.variable("a"))))) == 3  # and only in its body
assert ev(this.kind, this=E.OfLiteral.resolve(1)) == "literal"  # expressions are objects too
with raises(KeyError, match="variable 'x' is not bound"):
    ev(E.variable("x"))
with raises(TypeError, match="get expects an object, got int"):
    ev(E.operation("get", 1, "x"))
with raises(TypeError, match="has expects a property name, got int"):
    ev(E.operation("has", this, 1), this=ann)

## EVL-02 · Comparisons are three-valued and never coerce

In [ ]:
unit = E.literal(1)
assert [ev(unit.lt(2)), ev(unit.le(1)), ev(unit.gt(2)), ev(unit.ge(1)), ev(unit.eq(1)), ev(unit.ne(1))] == [
    True, True, False, True, True, False]
assert ev(E.literal("b").gt("a")) is True and ev(E.literal(b"\x01").lt(b"\x02")) is True
assert ev(unit.eq(1.0)) is None and ev(unit.lt(1.5)) is None and ev(unit.ne("1")) is None  # different types
assert ev(E.literal(True).eq(False)) is False and ev(E.literal(True).lt(False)) is None  # bools are not ordered
assert ev(E.literal(True).le(True)) is True
nan = E.literal(math.nan)
assert ev(nan.eq(math.nan)) is True and ev(nan.eq(1.0)) is False and ev(nan.lt(1.0)) is None
assert ev(E.literal(-0.0).lt(0.0)) is True and ev(E.literal(-0.0).eq(0.0)) is False
age = E.variable("x").age
assert ev(age.eq(1), x=nobody) is None and ev(age.ne(1), x=nobody) is None and ev(age.lt(1), x=nobody) is None
first, second = E.variable("a"), E.variable("b")
assert ev(first.eq(second), a=ann, b=ann) is True and ev(first.eq(second), a=ann, b=nobody) is False  # objects: by identity
assert ev(first.lt(second), a=ann, b=nobody) is None and ev(first.eq(1), a=ann) is None

## EVL-03 · Logic follows Kleene; arithmetic keeps one type

In [ ]:
T, F, U = E.literal(True), E.literal(False), E.variable("u")  # u is unknown


def table(method):
    return [[ev(getattr(x, method)(y), u=None) for y in (T, F, U)] for x in (T, F, U)]


assert table("and_") == [[True, False, None], [False, False, False], [None, False, None]]
assert table("or_") == [[True, True, True], [True, False, None], [True, None, None]]
assert table("implies") == [[True, False, None], [True, True, True], [True, None, None]]
assert [ev(x.not_(), u=None) for x in (T, F, U)] == [False, True, None]
missing = E.variable("missing")  # never evaluated when the first operand decides
assert ev(F.and_(missing)) is False and ev(T.or_(missing)) is True and ev(F.implies(missing)) is True
with raises(TypeError, match="and expects bool operands, got int"):
    ev(T.and_(1))
with raises(TypeError, match="not expects bool operands, got str"):
    ev(E.literal("x").not_())
pair = E.literal(2)
assert [ev(pair.add(3)), ev(pair.sub(3)), ev(pair.mul(3)), ev(pair.neg())] == [5, -1, 6, -2]
assert ev(E.literal(1.5).add(1.0)) == 2.5 and ev(E.literal(2**70).mul(2)) == 2**71
assert ev(U.add(1), u=None) is None and ev(U.neg(), u=None) is None
with raises(TypeError, match="add expects numbers of one type, got int and float"):
    ev(E.literal(1).add(1.5))
with raises(TypeError, match="sub expects numbers of one type, got bool and bool"):
    ev(T.sub(T))
with raises(TypeError, match="neg expects a number, got str"):
    ev(E.literal("x").neg())

## EVL-04 · Checking: `validate()` finds what evaluation would raise

In [ ]:
assert E.operation("lt", 1).data.validate() == ["lt takes 2 arguments, got 1"]
foo = E.operation("foo", 1).data
assert foo.validate() == [] and foo.validate(core=True) == ["'foo' is not a core operation"]
this = E.variable("this")
checked = E.let_("a", this.age, E.variable("a").ge(18).and_(this.has("email"))).data
assert checked.validate(bound={"this"}, core=True) == []
assert checked.validate() == ["value: argument 0: variable 'this' is not bound",
                              "body: argument 1: argument 0: variable 'this' is not bound"]
with raises(NotImplementedError, match="'foo' is not a core operation"):
    ev(foo)
with raises(TypeError, match="lt takes 2 arguments, got 1"):
    ev(E.operation("lt", 1))
with raises(ValueError, match="a literal needs a value"):
    ev(E.OfLiteral.Data())
with raises(ValueError, match="a let needs a value"):
    ev(E.OfLet.Data("x"))
with raises(ValueError, match="a let needs a body"):
    ev(E.OfLet.Data("x", E.OfLiteral.resolve(1)))
loop = E.OfOperation.Builder().name("not").create()
E.OfOperation.Builder(loop).arguments(loop).update()
with raises(ValueError, match="the expression contains a cycle"):
    ev(loop)
circular = E.OfLet.Data("a", None, E.OfLiteral.resolve(1))
circular.value = circular
assert circular.validate() == ["value: the expression contains a cycle"]
with raises(TypeError, match="not an expression: 7"):
    ev(E.OfOperation.Data("not", (7,)))

## EVL-05 · The union's discriminators pick each kind by its tag

In [ ]:
samples = [E.OfLiteral.resolve(1), op, E.OfVariable.resolve("x"), E.let_("a", 1, 2).data]
for branch, expected in zip(E.OfAny.Schema.branches, samples):
    assert branch.when.validate(bound={"this"}, core=True) == []
    assert [ev(branch.when, this=sample) for sample in samples] == [sample is expected for sample in samples]

## EVL-06 · Each kind has its own entry point, which accepts that kind's Specs

In [ ]:
x = E.variable("x")
assert V.OfLiteral(3) == 3 and V.OfLiteral(E.literal("s")) == "s"
assert V.OfVariable("x", {"x": 1}) == 1 and V.OfVariable(x, {"x": 2}) == 2
assert V.OfOperation(x.add(1), {"x": 1}) == 2 and V.OfLet(E.let_("y", 1, E.variable("y").neg())) == -1
assert V.OfAny(x, {"x": None}) is None and V.OfAny(lambda a: a.as_literal(1)) == 1
with raises(TypeError, match="expected a let or a callable taking its builder, got 3"):
    V.OfLet(3)
with raises(TypeError, match="expected an operation or a callable taking its builder"):
    V.OfOperation(x)  # a variable, not an operation